In [323]:
from pathlib import Path
from typing import List, Any
from langchain_community.document_loaders import Docx2txtLoader
from langchain_community.document_loaders.excel import UnstructuredExcelLoader
from langchain_community.document_loaders import PyPDFLoader, TextLoader, CSVLoader


class IngestionAgent:
    def load_all_docs(self,data_dir: str) -> List[Any]:
        """
        Load all supported files from the data directory 
        Convert to LangChain document structure.
        Supported: PDF, TXT, CSV, Excel, Word.
        """
        # Root data folder
        data_path = Path(data_dir).resolve()
        print(f"Status: Data path: {data_path}")
        
        #defined empty document list
        documents = []

        # PDF files
        pdf_files = list(data_path.glob('**/*.pdf'))
        print(f"Status: Found {len(pdf_files)} PDF files: {[str(f) for f in pdf_files]}")
        for pdf_file in pdf_files:
            print(f"Status: Loading PDF: {pdf_file}")
            try:
                loader = PyPDFLoader(str(pdf_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} PDF docs from {pdf_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load PDF {pdf_file}: {e}")

        # Word files
        docx_files = list(data_path.glob('**/*.docx'))
        print(f"Status: Found {len(docx_files)} Word files: {[str(f) for f in docx_files]}")
        for docx_file in docx_files:
            print(f"Status: Loading Word: {docx_file}")
            try:
                loader = Docx2txtLoader(str(docx_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} Word docs from {docx_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load Word {docx_file}: {e}")

        # CSV files
        csv_files = list(data_path.glob('**/*.csv'))
        print(f"Status: Found {len(csv_files)} CSV files: {[str(f) for f in csv_files]}")
        for csv_file in csv_files:
            print(f"Status: Loading CSV: {csv_file}")
            try:
                loader = CSVLoader(str(csv_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} CSV docs from {csv_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load CSV {csv_file}: {e}")

        # Excel files
        xlsx_files = list(data_path.glob('**/*.xlsx'))
        print(f"Status: Found {len(xlsx_files)} Excel files: {[str(f) for f in xlsx_files]}")
        for xlsx_file in xlsx_files:
            print(f"Status: Loading Excel: {xlsx_file}")
            try:
                loader = UnstructuredExcelLoader(str(xlsx_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} Excel docs from {xlsx_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load Excel {xlsx_file}: {e}")

        # TXT files
        txt_files = list(data_path.glob('**/*.txt'))
        print(f"Status: Found {len(txt_files)} TXT files: {[str(f) for f in txt_files]}")
        for txt_file in txt_files:
            print(f"Status: Loading TXT: {txt_file}")
            try:
                loader = TextLoader(str(txt_file))
                loaded = loader.load()
                print(f"Status: Loaded {len(loaded)} TXT docs from {txt_file}")
                documents.extend(loaded)
            except Exception as e:
                print(f"[ERROR] Failed to load TXT {txt_file}: {e}")

        
        print(f"Status: Total loaded documents: {len(documents)}")
        return documents

# Example usage
if __name__ == "__main__":
    ingest=IngestionAgent()
    docs = ingest.load_all_docs("./data")
    print(f"Loaded {len(docs)} documents.")
    print("Example document:", docs[0] if docs else None)

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 150 0 (offset 0)
Ignoring wrong pointing object 151 0 (offset 0)


Status: Data path: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data
Status: Found 2 PDF files: ['/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf', '/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf']
Status: Loading PDF: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf
Status: Loaded 18 PDF docs from /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf
Status: Loading PDF: /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf
Status: Loaded 15 PDF docs from /Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/attention all you need.pdf
Status: Found 1 Word files: ['/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/docx/Project Proposal Smart Travel-Adventure Planner Bot.docx']
Status: Loading Word: /Users/pavankumarb/Documents/My Learning/DocEN

In [320]:
from typing import List, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
#from agents.ingestion_agent import load_all_docs

class Embedding:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model = SentenceTransformer(model_name)
        print(f"Status: Loaded embedding model: {model_name}")

    def chunk_documents(self, documents: List[Any]) -> List[Any]:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            length_function=len,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = splitter.split_documents(documents)
        print(f"Status: Split {len(documents)} documents into {len(chunks)} chunks.")
        return chunks

    def embed_chunks(self, chunks: List[Any]) -> np.ndarray:
        texts = [chunk.page_content for chunk in chunks]
        print(f"Status: Generating embeddings for {len(texts)} chunks...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Status: Embeddings shape: {embeddings.shape}")
        return embeddings

# Example usage
if __name__ == "__main__":
    emb_pipe = Embedding()
    chunks = emb_pipe.chunk_documents(docs)
    embeddings = emb_pipe.embed_chunks(chunks)
    print("Status: Example embedding:", chunks[0] if len(chunks) > 0 else None ,embeddings[0] if len(embeddings) > 0 else None)

Status: Loaded embedding model: all-MiniLM-L6-v2
Status: Split 9835 documents into 13058 chunks.
Status: Generating embeddings for 13058 chunks...


Batches: 100%|██████████| 409/409 [00:20<00:00, 19.92it/s]


Status: Embeddings shape: (13058, 384)
Status: Example embedding: page_content='Data 
Protection 
and Privacy
Group 4
1.Pavan Kumar Boddupally 
2.Kusuma Kankanala 
3.Kevin Rodriguez 
4. Md Istihad Alam 
5. Benjamin Castro' metadata={'producer': 'macOS Version 15.6.1 (Build 24G90) Quartz PDFContext', 'creator': 'Keynote', 'creationdate': "D:20251122123256Z00'00'", 'title': 'Group 4 – Data Protection and Privacy.pdf', 'author': 'Pavan Kumar Boddupally', 'moddate': "D:20251122123256Z00'00'", 'source': '/Users/pavankumarb/Documents/My Learning/DocENTmcp/data/pdf/Group 4 – Data Protection and Privacy.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'} [-4.30258326e-02  3.00627183e-02 -3.32171805e-02 -5.40830940e-02
  3.85903120e-02  5.35517447e-02  7.38651529e-02 -4.47132513e-02
  2.43362207e-02  1.60339754e-02  1.03239611e-01 -7.51614664e-03
  2.98262481e-02 -7.35344961e-02 -1.27182361e-02  4.44146283e-02
 -2.32653487e-02  1.46100251e-02 -3.63918580e-02 -7.91264251e-02
 -6.64726272e-02 

In [ ]:
import os
import faiss
import numpy as np
import pickle
from typing import List, Any
from sentence_transformers import SentenceTransformer
#from src.embedding import EmbeddingPipeline

class FaissVectorStore:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", chunk_size: int = 1000, chunk_overlap: int = 200):
        self.persist_dir = persist_dir
        os.makedirs(self.persist_dir, exist_ok=True)
        self.index = None
        self.metadata = []
        self.embedding_model = embedding_model
        self.model = SentenceTransformer(embedding_model)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        print(f"[INFO] Loaded embedding model: {embedding_model}")

    def build_from_documents(self, documents: List[Any]):
        print(f"[INFO] Building vector store from {len(documents)} raw documents...")
        emb_pipe = EmbeddingPipeline(model_name=self.embedding_model, chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap)
        chunks = emb_pipe.chunk_documents(documents)
        embeddings = emb_pipe.embed_chunks(chunks)
        metadatas = [{"text": chunk.page_content} for chunk in chunks]
        self.add_embeddings(np.array(embeddings).astype('float32'), metadatas)
        self.save()
        print(f"[INFO] Vector store built and saved to {self.persist_dir}")

    def add_embeddings(self, embeddings: np.ndarray, metadatas: List[Any] = None):
        dim = embeddings.shape[1]
        if self.index is None:
            self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings)
        if metadatas:
            self.metadata.extend(metadatas)
        print(f"[INFO] Added {embeddings.shape[0]} vectors to Faiss index.")

    def save(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        faiss.write_index(self.index, faiss_path)
        with open(meta_path, "wb") as f:
            pickle.dump(self.metadata, f)
        print(f"[INFO] Saved Faiss index and metadata to {self.persist_dir}")

    def load(self):
        faiss_path = os.path.join(self.persist_dir, "faiss.index")
        meta_path = os.path.join(self.persist_dir, "metadata.pkl")
        self.index = faiss.read_index(faiss_path)
        with open(meta_path, "rb") as f:
            self.metadata = pickle.load(f)
        print(f"[INFO] Loaded Faiss index and metadata from {self.persist_dir}")

    def search(self, query_embedding: np.ndarray, top_k: int = 5):
        D, I = self.index.search(query_embedding, top_k)
        results = []
        for idx, dist in zip(I[0], D[0]):
            meta = self.metadata[idx] if idx < len(self.metadata) else None
            results.append({"index": idx, "distance": dist, "metadata": meta})
        return results

    def query(self, query_text: str, top_k: int = 5):
        print(f"[INFO] Querying vector store for: '{query_text}'")
        query_emb = self.model.encode([query_text]).astype('float32')
        return self.search(query_emb, top_k=top_k)

# Example usage
if __name__ == "__main__":
    store = FaissVectorStore("faiss_store")
    store.build_from_documents(docs)
    store.load()
    print(store.query("What is attention mechanism?", top_k=3))
    print()

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Building vector store from 9835 raw documents...
Status: Loaded embedding model: all-MiniLM-L6-v2
Status: Split 9835 documents into 13058 chunks.
Status: Generating embeddings for 13058 chunks...


Batches: 100%|██████████| 409/409 [00:22<00:00, 18.44it/s]


Status: Embeddings shape: (13058, 384)
[INFO] Added 13058 vectors to Faiss index.
[INFO] Saved Faiss index and metadata to faiss_store
[INFO] Vector store built and saved to faiss_store
[INFO] Loaded Faiss index and metadata from faiss_store
[INFO] Querying vector store for: 'What is attention mechanism?'
[{'index': np.int64(29), 'distance': np.float32(0.7285826), 'metadata': {'text': '3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'}}, {'index': np.int64(66), 'distance': np.float32(0.8640241), 'metadata': {'text': 'Attention Visualizations\nInput-Input Layer5\nIt\nis\nin\nthis\nspirit\nthat\na\nmajority\nof\nAmerican\ngovernments\nhave\npassed\nnew\nlaws\nsince\n2009\nmaking\nthe\nregistration\nor\nvoting\nprocess\nmore\ndifficult\n.\n<EOS>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\n<pad>\nIt\nis\nin\nthis\nspirit\nthat\na

In [333]:
import os
from dotenv import load_dotenv
#from src.vectorstore import FaissVectorStore
#from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama

class RAGSearch:
    def __init__(self, persist_dir: str = "faiss_store", embedding_model: str = "all-MiniLM-L6-v2", llm_model: str = "gemma2-9b-it"):
        self.vectorstore = FaissVectorStore(persist_dir, embedding_model)
        # Load or build vectorstore
        faiss_path = os.path.join(persist_dir, "faiss.index")
        meta_path = os.path.join(persist_dir, "metadata.pkl")
        if not (os.path.exists(faiss_path) and os.path.exists(meta_path)):
            #from data_loader import load_all_documents
            ingest=IngestionAgent()
            docs = ingest.load_all_docs("data")
            self.vectorstore.build_from_documents(docs)
        else:
            self.vectorstore.load()
        #groq_api_key = ""
        self.llm = ChatOllama(model="llama3.1")
        print(f"Status Ollama LLM initialized: {self.llm}")

    def search_and_summarize(self, query: str, top_k: int = 5) -> str:
        results = self.vectorstore.query(query, top_k=top_k)
        texts = [r["metadata"].get("text", "") for r in results if r["metadata"]]
        context = "\n\n".join(texts)
        if not context:
            return "No relevant documents found."
        prompt = f"""Summarize the following context for the query: '{query}'\n\nContext:\n{context}\n\nSummary:"""
        response = self.llm.invoke([prompt])
        return response.content

# Example usage
if __name__ == "__main__":
    

    rag_search = RAGSearch()
    query = "Explain about model context protocol in 5 points ?"
    summary = rag_search.search_and_summarize(query, top_k=3)
    print("Summary:", summary)
    

[INFO] Loaded embedding model: all-MiniLM-L6-v2
[INFO] Loaded Faiss index and metadata from faiss_store
Status Ollama LLM initialized: model='llama3.1'
[INFO] Querying vector store for: 'Explain about model context protocol in 5 points ?'
Summary: Here is a summary of the Model Context Protocol in 5 points:

1. **Key Components**: The Model Context Protocol consists of several key components, including Base Protocol, Lifecycle Management, Authorization, Server Features, Client Features, and Utilities.
2. **Implementation Requirements**: All implementations MUST support the base protocol and lifecycle management components, while other components MAY be implemented based on specific application needs.
3. **Modular Design**: The modular design allows for clear separation of concerns and enables rich interactions between clients and servers, supporting exactly the features needed by each implementation.
4. **Reserved Key Names**: Certain key names are reserved by MCP for protocol-level me

1+1